# Language Subtitles — Burn

Third and final step of Language Subtitles: burns the final per-language subtitles (from
`caption-multilang-generate.ipynb`) onto the video base — one line per language, stacked, each
in its own color (same palette as `config.CORES_IDIOMAS`).

Always reads the current file from Drive for each language — if you downloaded a subtitle,
corrected it, and re-uploaded it, this notebook picks up your correction automatically.

Output: `{NOME}_final_idiomas.mp4` — a **different** file from Single Subtitle's
`{NOME}_final.mp4`, so both can coexist in the same video folder.

This is the simple burn mode (solid color per language) — not the word-by-word morphological
classification mode, which is a separate, currently inactive stage.


---

> ### 🇨🇳 Variante de 6 idiomas (com chinês)
> Cópia do notebook de 5 idiomas com o chinês (`zh`) incluído — o original
> continua intacto e funcionando. As duas variantes convivem na mesma pasta
> do vídeo: esta grava os finais com sufixo `_zh`
> (`<nome>_final_idiomas_zh.mp4`, `<nome>_final_multicolor_zh.mp4`), então
> rodar esta **não** sobrescreve o resultado de 5 idiomas.
>
> Convenção de código do chinês: `zh` internamente (posição na tela, cor,
> fonte) · `zh-Hans` no YouTube · `zh-hans` no Stanza. Simplificado, não
> tradicional — é o usado na China continental, Singapura e Malásia.


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  📦 SETUP — packages, Google Drive, and modules (run once per session) ║
# ╚══════════════════════════════════════════════════════════════════╝

!apt-get -qq -y install ffmpeg fonts-noto-cjk > /dev/null 2>&1
print('✅ ffmpeg + fonts-noto-cjk (needed to render Korean/CJK subtitles — without')
print('   this font installed, CJK text renders as blank space, even though the')
print('   language label still shows up)')

from google.colab import drive
try:
    drive.flush_and_unmount()
except Exception:
    pass
drive.mount('/content/drive', force_remount=True)
print('✅ Drive mounted')

import shutil, os, sys, logging
from pathlib import Path

PASTA_DRIVE_RAIZ = "narrated_video"
PASTA_MODULOS = Path(f"/content/drive/MyDrive/{PASTA_DRIVE_RAIZ}/pipeline/modulos")
DESTINO = Path("/content/pipeline")

if PASTA_MODULOS.exists():
    if DESTINO.exists():
        shutil.rmtree(DESTINO)
    shutil.copytree(PASTA_MODULOS, DESTINO)
    print(f"✅ {len(list(DESTINO.glob('*.py')))} modules copied from {PASTA_MODULOS}")
else:
    print(f"❌ Modules folder not found: {PASTA_MODULOS}")

if str(DESTINO) not in sys.path:
    sys.path.insert(0, str(DESTINO))

# ── Clear stale local data files from any previous run in this session ─────
# Modules above are always freshly copied (rmtree + copytree), but DATA files
# (.srt, .ass, .mp4) downloaded by earlier cells in this same session could
# still be sitting in /content — if you re-run after correcting something on
# Drive, you want THIS run to re-download everything fresh, not silently
# reuse an old local copy. This removes any leftover language-subtitle data
# files before starting.
os.chdir('/content')
padroes_para_limpar = ["*.srt", "*.ass", "*_final_idiomas.mp4"]
removidos = 0
for padrao in padroes_para_limpar:
    for arquivo in Path('/content').glob(padrao):
        arquivo.unlink()
        removidos += 1
print(f"✅ {removidos} stale local file(s) cleared — this run will fetch everything fresh from Drive")

logging.basicConfig(level=logging.INFO, format='%(asctime)s  %(name)-24s  %(levelname)s  %(message)s', datefmt='%H:%M:%S')
print('✅ Setup complete!')


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  ⚙️  CONFIGURATION                                               ║
# ║  ✏️  Edit only this cell — same NOME_ORACAO as earlier steps     ║
# ╚══════════════════════════════════════════════════════════════════╝

# ── 1. VIDEO IDENTITY ────────────────────────────────────────────────────────
NOME_ORACAO = "40_Matt_02"

# ── 2. MASTER LANGUAGE (must match caption-single-generate.ipynb / caption-multilang-generate.ipynb) ──
# If IDIOMA_MESTRE is included in IDIOMAS_ALVO below, the pipeline uses
# NOME_LEGENDA_MESTRE directly for it — no need to duplicate/rename that
# file, the master caption already IS that language's subtitle.
IDIOMA_MESTRE = "en"
NOME_LEGENDA_MESTRE = "40_Matt_02_whisper_en.srt"

# ── 3. LANGUAGES TO BURN (must have been generated already) ────────────────
IDIOMAS_ALVO = ["en", "pt", "es", "fr", "ko", "zh"]

# ── 4. VERSE REFERENCE OVERLAY (optional) ───────────────────────────────────
# A small fixed indicator in the top-left corner (e.g. "Matt/Mt/마 2:4") that
# updates only the verse number as the narration advances — separate from
# the language subtitles stacked in the middle. OPTIONAL: only makes sense
# for verse-by-verse Bible study videos — leave INCLUIR_VERSICULO = False
# for prayer/free-content videos that aren't tied to Bible verses.
INCLUIR_VERSICULO = True

# Only used if INCLUIR_VERSICULO = True:

# CAPITULO: leave None to derive it from NOME_ORACAO — "40_Matt_02" already
# says the chapter is 2. The old default was a hard-coded 1, which on a video
# named 40_Matt_02 burned "Matt 1:4" over chapter 2 with no error at all. Only
# fill this in for a video whose name does NOT follow {NN}_{Sigla}_{CC}.
CAPITULO = None

ABREVIACOES_LIVRO = {"en": "Matt", "pt": "Mt", "es": "Mt", "fr": "Mt", "ko": "마", "zh": "太"}

# TEXTO_VERSICULOS: leave "" and the notebook finds the text itself — the
# video's roteiro_versiculos.txt on Drive, or the whole Bible in
# dados_lexico/web-biblia.json. Fill this in only to use a text different
# from both (verse numbers as standalone tokens in the flow:
# "1 Now when Jesus was born ... 2 Where is he who is born ...").
TEXTO_VERSICULOS = ""

# ── 5. DRIVE ROOT FOLDER ────────────────────────────────────────────────────
PASTA_DRIVE_RAIZ = "narrated_video"     # ⚠️ DO NOT CHANGE — fixed for the whole project

# ── CHECK ──────────────────────────────────────────────────────────────────
print("=" * 60)
print("⚙️  CONFIGURATION")
print("=" * 60)
print(f"   Video:            {NOME_ORACAO}")
print(f"   Master language:  {IDIOMA_MESTRE}  ({NOME_LEGENDA_MESTRE})")
print(f"   Languages:        {IDIOMAS_ALVO}")
print(f"   Verse overlay:    {'ON — ' + '/'.join(dict.fromkeys(ABREVIACOES_LIVRO.values())) + f' {CAPITULO}:N' if INCLUIR_VERSICULO else 'off'}")
print(f"   Drive root:       {PASTA_DRIVE_RAIZ}")
print("=" * 60)
print("✅ Configuration ready — proceed to Initialization")


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  🚀 INITIALIZE PIPELINE                                          ║
# ╚══════════════════════════════════════════════════════════════════╝

import sys
from pathlib import Path

if '/content/pipeline' not in sys.path:
    sys.path.insert(0, '/content/pipeline')

from config import PipelineConfig
from language_captions_pipeline import LanguageCaptionsPipeline

# O nome do vídeo já carrega livro e capítulo — não peça duas vezes o que
# `40_Matt_02` só pode significar. Recusa nome fora do padrão em vez de
# adivinhar: capítulo adivinhado errado não dá erro, só sai no vídeo.
if CAPITULO is None:
    import biblia_livros as _bl
    _livro, CAPITULO = _bl.de_nome_projeto(NOME_ORACAO)
    print(f"📖 Chapter derived from {NOME_ORACAO}: {_livro.nome} {CAPITULO}")


# Fundo: "imagem", "video", ou None pra detectar sozinho pelo arquivo que
# existe no Drive. Só preencha à mão se as duas versões estiverem lá.
MODO_CLIPE = None

# O modo do fundo (imagem parada ou clipe de vídeo) sai do arquivo que EXISTE
# no Drive, não de uma opção que dá pra esquecer de marcar -- mesma ideia do
# sufixo `_zh` lido do nome do .ass. Sem isto, queimar sobre um vídeo base
# feito em modo imagem procurava `_video_base.mp4` e não achava; e se achasse,
# o resultado sairia sem `_img`, por cima da versão de clipe.
import config as _cfgmod
if MODO_CLIPE is None:
    MODO_CLIPE = _cfgmod.detectar_modo_clipe(
        Path(f"/content/drive/MyDrive/{PASTA_DRIVE_RAIZ}/videos/{NOME_ORACAO}"),
        NOME_ORACAO,
    )
    print(f"🎞️  Background mode detected from Drive: {MODO_CLIPE}")

config = PipelineConfig(
    NOME_ORACAO           = NOME_ORACAO,
    PASTA_DRIVE_RAIZ       = PASTA_DRIVE_RAIZ,
    IDIOMA_MESTRE          = IDIOMA_MESTRE,
    NOME_LEGENDA_MESTRE    = NOME_LEGENDA_MESTRE,
    MODO_CLIPE             = MODO_CLIPE,
    CAPITULO               = CAPITULO,
    ABREVIACOES_LIVRO      = ABREVIACOES_LIVRO,
    SUFIXO_VARIANTE_IDIOMAS = "_zh",   # -> <nome>_final_idiomas_zh.mp4
)

pipeline = LanguageCaptionsPipeline(config)  # no GroqClient needed for burning

print("=" * 60)
print("✅ PIPELINE INITIALIZED")
print("=" * 60)
print(f"   Video:          {config.NOME_ORACAO}")
print(f"   Folder:         {config.pasta_oracao}")
print(f"   Master caption: {config.nome_legenda_mestre}")
print(f"   Output video:   {config.NOME_VIDEO_FINAL_IDIOMAS}")
print("=" * 60)


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  📖 GENERATE VERSE REFERENCE (optional — only if INCLUIR_VERSICULO) ║
# ║  Skipped automatically if INCLUIR_VERSICULO = False.              ║
# ╚══════════════════════════════════════════════════════════════════╝

if INCLUIR_VERSICULO:
    # Três fontes, da mais específica pra mais geral (ver
    # caption_pipeline.resolver_texto_versiculos): o que você colou acima, o
    # roteiro do vídeo no Drive, ou o web-biblia.json com a Bíblia inteira.
    from caption_pipeline import resolver_texto_versiculos

    TEXTO_VERSICULOS, _origem = resolver_texto_versiculos(config, TEXTO_VERSICULOS)
    print(f"📖 Verse text from {_origem} ({len(TEXTO_VERSICULOS)} chars)")

    srt_versiculo = pipeline.gerar_legenda_versiculo(TEXTO_VERSICULOS)
    print(f"✅ Verse reference generated: {srt_versiculo.name}")
else:
    print("Verse reference overlay is off (INCLUIR_VERSICULO = False) — skipping.")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  🔥 LOAD + BURN — language subtitles stacked onto the video base ║
# ║  Always downloads the current files from Drive — picks up any    ║
# ║  manual correction automatically.                                 ║
# ╚══════════════════════════════════════════════════════════════════╝

legendas_idiomas = pipeline.carregar_idiomas_finais(IDIOMAS_ALVO)
print(f"{len(legendas_idiomas)}/{len(IDIOMAS_ALVO)} languages loaded: {list(legendas_idiomas.keys())}\n")

video_final = pipeline.queimar_idiomas(legendas_idiomas, incluir_versiculo=INCLUIR_VERSICULO)
print(f"\n✅ Final video: {video_final.name} ({video_final.stat().st_size/1_048_576:.1f} MB)")


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  👀 PREVIEW FINAL VIDEO                                          ║
# ╚══════════════════════════════════════════════════════════════════╝

from IPython.display import Video, display

display(Video(str(video_final), embed=True, width=800))


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  📥 DOWNLOAD — FINAL VIDEO                                       ║
# ╚══════════════════════════════════════════════════════════════════╝

from google.colab import files

print(f"📥 Downloading {video_final.name} ({video_final.stat().st_size/1_048_576:.1f} MB)...")
files.download(str(video_final))
